# 5.20 MCMC: Metropolis-Hastings & Gibbs

**Lesson tagline.** MCMC trades independent samples for a Markov chain whose stationary distribution is the posterior.

MCMC builds on sampling when direct proposals are hard. We implement Metropolis-Hastings and Gibbs on a D1-D5 ladder from tiny discrete targets to a poorly mixing high-dimensional posterior. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 520
rng = np.random.default_rng(SEED)

def normalize(values):
    arr = np.asarray(values, dtype=float)
    total = arr.sum()
    if total <= 0:
        raise ValueError("positive mass required")
    return arr / total

def effective_sample_size(series):
    x = np.asarray(series, dtype=float)
    x = x - x.mean()
    denom = np.dot(x, x)
    if denom == 0:
        return float(len(x))
    max_lag = min(200, len(x) - 1)
    autocorr = []
    for lag in range(1, max_lag):
        value = np.dot(x[:-lag], x[lag:]) / denom
        if value <= 0:
            break
        autocorr.append(value)
    tau = 1.0 + 2.0 * np.sum(autocorr)
    return len(x) / tau

def total_variation(empirical, truth):
    empirical = normalize(empirical)
    truth = normalize(truth)
    return 0.5 * np.sum(np.abs(empirical - truth))

def build_mcmc_ladder():
    ladder = []
    ladder.append({"name": "D1 2-state", "kind": "discrete", "target": np.array([0.25, 0.75])})
    ladder.append({"name": "D2 4-state lesson", "kind": "discrete", "target": np.array([0.1, 0.2, 0.4, 0.3])})
    ladder.append({"name": "D3 bimodal valley", "kind": "discrete", "target": normalize([0.42, 0.05, 0.06, 0.05, 0.42])})
    ladder.append({"name": "D4 correlated 2-D", "kind": "gaussian", "mean": np.zeros(2), "cov": np.array([[1.0, 0.92], [0.92, 1.2]])})
    ladder.append({"name": "D5 ill-conditioned 8-D", "kind": "gaussian", "mean": np.zeros(8), "cov": np.diag([0.03, 0.05, 0.1, 0.2, 0.8, 1.5, 3.0, 6.0])})
    return ladder

def log_gaussian(x, mean, cov):
    x = np.asarray(x)
    diff = x - mean
    inv = np.linalg.inv(cov)
    return -0.5 * float(diff @ inv @ diff)

def run_discrete_chain(target, n_steps, random_state, start=0, asymmetric=False):
    target = normalize(target)
    state = int(start)
    states = []
    accepted = 0
    for _ in range(n_steps):
        if asymmetric:
            step = 1 if random_state.random() < 0.7 else -1
        else:
            step = 1 if random_state.random() < 0.5 else -1
        proposal = (state + step) % len(target)
        forward = 0.7 if asymmetric and step == 1 else 0.3 if asymmetric else 0.5
        reverse_step = -step
        reverse = 0.7 if asymmetric and reverse_step == 1 else 0.3 if asymmetric else 0.5
        ratio = target[proposal] * reverse / (target[state] * forward)
        if random_state.random() < min(1.0, ratio):
            state = proposal
            accepted += 1
        states.append(state)
    return np.array(states), accepted / n_steps

def run_gaussian_rw(rung, n_steps, step_scale, random_state):
    x = np.zeros(len(rung["mean"]))
    states = []
    accepted = 0
    current = log_gaussian(x, rung["mean"], rung["cov"])
    for _ in range(n_steps):
        proposal = x + random_state.normal(0.0, step_scale, size=len(x))
        proposed = log_gaussian(proposal, rung["mean"], rung["cov"])
        if np.log(random_state.random()) < proposed - current:
            x = proposal
            current = proposed
            accepted += 1
        states.append(x.copy())
    return np.array(states), accepted / n_steps

## The concept, built once (D1)
Metropolis-Hastings accepts a proposed move with
$$a(x\to x^\prime)=\min\left(1,\frac{\pi(x^\prime)q(x\mid x^\prime)}{\pi(x)q(x^\prime\mid x)}\right).$$
The implementation keeps the proposal ratio explicit so asymmetric kernels are not accidentally treated as symmetric.

In [ ]:
def metropolis_hastings_step(current, proposal, pi, q_forward, q_reverse, random_state):
    ratio = pi[proposal] * q_reverse / (pi[current] * q_forward)
    acceptance = min(1.0, ratio)
    if random_state.random() < acceptance:
        return proposal, acceptance, True
    return current, acceptance, False

lesson_pi = np.array([0.1, 0.2, 0.4, 0.3])
new_state, acceptance, accepted = metropolis_hastings_step(1, 2, lesson_pi, 0.5, 0.5, rng)
assert acceptance == 1.0
assert new_state == 2
print("acceptance for 1 -> 2", acceptance)

Gibbs sampling is the conditional special case. If the pairwise score has odds $4:1$ for $X=1$ versus $X=0$ when $Y=1$, then $p(X=1\mid Y=1)=4/(1+4)=0.800$.

In [ ]:
def gibbs_conditional_x(y_value, table):
    column = table[:, y_value]
    return normalize(column)

pair_table = np.array([[1.0, 1.0], [1.0, 4.0]])
cond = gibbs_conditional_x(1, pair_table)
assert abs(cond[1] - 0.8) < 1e-12
assert np.allclose(lesson_pi, [0.1, 0.2, 0.4, 0.3])
print("p(X=1 | Y=1)", cond[1])

## The dataset ladder
The F10 ladder moves from exact discrete targets to a bimodal valley, then to correlated and high-dimensional Gaussian targets where random-walk mixing becomes the limiting factor.

In [ ]:
ladder = build_mcmc_ladder()
for i, rung in enumerate(ladder, start=1):
    print(i, rung["name"], rung["kind"])

## Run the SAME method across D1-D5
Collect the plan metric on every rung from D1–D5 with the same algorithmic implementation, then print a compact per-rung table.

In [ ]:
checkpoints = [50, 100, 250, 500, 1000, 2000]
results = {}
print("rung | final TV or marginal error | ESS | acceptance")
for rung in ladder:
    if rung["kind"] == "discrete":
        states, accept_rate = run_discrete_chain(rung["target"], checkpoints[-1], rng)
        errors = []
        for n in checkpoints:
            empirical = np.bincount(states[:n], minlength=len(rung["target"]))
            errors.append(total_variation(empirical, rung["target"]))
        ess = effective_sample_size(states == np.argmax(rung["target"]))
        results[rung["name"]] = {"states": states, "errors": errors, "ess": ess, "accept": accept_rate}
    else:
        states, accept_rate = run_gaussian_rw(rung, checkpoints[-1], 0.45, rng)
        errors = []
        truth = rung["mean"][0]
        for n in checkpoints:
            errors.append(abs(states[:n, 0].mean() - truth))
        ess = effective_sample_size(states[:, 0])
        results[rung["name"]] = {"states": states, "errors": errors, "ess": ess, "accept": accept_rate}
    print(f"{rung['name']} | {results[rung['name']]['errors'][-1]:.4f} | {results[rung['name']]['ess']:.1f} | {results[rung['name']]['accept']:.2f}")

## Results visualization
The closing figure has target/sample panels on top and the requested error-vs-iteration or error-vs-sample curve below.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, rung in zip(axes[0], ladder):
    info = results[rung["name"]]
    states = info["states"]
    if rung["kind"] == "discrete":
        counts = np.bincount(states, minlength=len(rung["target"]))
        empirical = counts / counts.sum()
        xs = np.arange(len(rung["target"]))
        ax.bar(xs - 0.15, rung["target"], width=0.3)
        ax.bar(xs + 0.15, empirical, width=0.3)
    else:
        ax.hist(states[:, 0], bins=30, density=True, alpha=0.75)
        ax.axvline(rung["mean"][0], color="black", linestyle="--")
    ax.set_title(rung["name"], fontsize=8)
for ax, rung in zip(axes[1], ladder):
    ax.plot(checkpoints, results[rung["name"]]["errors"], marker="o")
    ax.set_xscale("log")
    ax.set_title("error vs iteration", fontsize=8)
fig.tight_layout()
plt.show()

## Pitfall on the hardest rung
On D5, ignoring burn-in and mixing makes early states dominate, and dropping proposal asymmetry biases discrete chains. The fix is to include the $q$ ratio, discard burn-in, and use a proposal scale that actually moves through the narrow dimensions.

In [ ]:
d5 = ladder[-1]
bad_states, bad_accept = run_gaussian_rw(d5, 2500, 1.2, rng)
good_states, good_accept = run_gaussian_rw(d5, 2500, 0.18, rng)
bad_error = abs(bad_states[:, 0].mean() - d5["mean"][0])
good_error = abs(good_states[500:, 0].mean() - d5["mean"][0])
print("bad no burn-in / poor scale", round(float(bad_error), 4), "accept", round(bad_accept, 3))
print("fixed burn-in / tuned scale", round(float(good_error), 4), "accept", round(good_accept, 3))
asymmetric_states, _ = run_discrete_chain(lesson_pi, 3000, rng, asymmetric=True)
tv_asymmetric = total_variation(np.bincount(asymmetric_states, minlength=4), lesson_pi)
print("asymmetric chain with correct q-ratio TV", round(float(tv_asymmetric), 4))

## Evaluate it + Practice
- **Metric.** Total-variation distance for discrete rungs; marginal error and ESS for continuous rungs.
- **No-skill baseline.** Use the initial state or proposal distribution as the posterior estimate.
- **Cheap sanity check.** Acceptance rates should be neither zero nor one for nontrivial random-walk proposals.
- **Ablation.** Remove burn-in or omit the proposal ratio in an asymmetric chain.
- **Failure signals.** Sticky traces, ESS near one, or empirical frequencies far from the target after many iterations.

### Practice prompts
1. Change the D3 valley mass and compare TV curves.
2. Implement a coordinate-wise Gibbs sampler for the D4 Gaussian.
3. Run two chains from overdispersed starts and compare their marginal means.

In [ ]:
# Your code here

In [ ]:
# Your code here

In [ ]:
# Your code here